In [17]:
# ruff: noqa
import sys, os
sys.path.append(os.path.abspath("./../feedback-grape"))
sys.path.append(os.path.abspath("./../"))

# ruff: noqa
from feedback_grape.fgrape import optimize_pulse
from helpers import (
    init_fgrape_protocol,
    test_implementations,
    generate_superposition_state,
    experiment_param_formats,
    generate_povm,
)
from library.utils.FgResult_to_dict import FgResult_to_dict
import json, jax, time
import jax.numpy as jnp

test_implementations()

In [18]:
batch_size = 1
N_qubits = 2
base_dim = 2**N_qubits
N_povm_params = base_dim*(base_dim+1)
key = jax.random.PRNGKey(42)
povm_params_batch = jax.random.uniform(key, (batch_size, N_povm_params), minval=0.0, maxval=2*jnp.pi)

@jax.jit
def f(povm_params_batch):
    def f2(povm_params):
        sum = 0.0
        for _ in range(10):
            povm = generate_povm(+1, params=povm_params, dim=base_dim)
            sum = sum + jnp.abs(jnp.sum(povm))
        return sum
    
    povm_params_batch = jax.vmap(f2)(povm_params_batch)
    return jnp.sum(povm_params_batch)

f(povm_params_batch)

start = time.time()
for _ in range(100):
    f(povm_params_batch)
print("Time taken:", time.time() - start)

start = time.time()
for _ in range(100):
    value, grad = jax.value_and_grad(f)(povm_params_batch)
print("Time taken:", time.time() - start)

Time taken: 0.05169034004211426
Time taken: 11.49152135848999


In [19]:
N = 10000
batch_size = 1
arr = jnp.arange(N, dtype=jnp.float64)
arr_batch = jnp.tile(arr, (batch_size, 1))

@jax.jit
def f(arr_batch):
    def f2(arr):
        sum = arr[0:N//2] + arr[N//2:N]
        for i in range(100):
            sum = sum + arr[0:N//2]
            sum = sum + arr[N//2:N]
        return sum
    
    arr_batch = jax.vmap(f2)(arr_batch)

    return jnp.sum(arr_batch)

f(arr_batch)

start = time.time()
for _ in range(100):
    f(arr_batch)
print("Time taken:", time.time() - start)

start = time.time()
for _ in range(100):
    value, grad = jax.value_and_grad(f)(arr_batch)
print("Time taken:", time.time() - start)

Time taken: 0.004278421401977539
Time taken: 1.0256612300872803
